# CSCI 443 Lecture 22 Spark Performance Evaluation II

We are going to study the performance of orderBy which invokes a Spark full sort.

## Create a random dataset

We create a column of random numbers spanning 8 partitions.  

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = SparkSession.builder.getOrCreate()

# Make shuffle stages visible without creating too many tiny tasks.
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Generate random floating-point values.
n = 10_000_000

index_df = spark.range(n)  # column of n rows where the first row contains 0, 2nd contains 1, etc.

# Create a DataFrame wtih a single column named `x` that contains random floating-point values.
df = index_df.select(rand(seed=42).alias("x"))

# Force materialization of the unsorted dataset.
df.count()

df.write.mode("overwrite").saveAsTable("random_float_unsorted")


In [0]:
df = spark.table("random_float_unsorted")
display(df.limit(10))

## Sort 

We use `orderBy` to sort.

In [0]:


# Global full sort.
sorted_df = df.orderBy("x")

# Show the physical plan.
sorted_df.explain("formatted")


# Trigger execution so Spark creates stages you can inspect in the Spark UI.
sorted_df.write.mode("overwrite").saveAsTable("random_float_sorted")
#sorted_df.count()  # count may not be enough because Spark doesn't need ordering to count.  It may optimize away the sort.

In [0]:
sorted_df = df.orderBy("x")
sorted_df.write.mode("overwrite").saveAsTable("random_float_sorted")
